# Classification

| Key              | Value                                                                                                                                                                                                                                                                                                 |
|:-----------------|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Course Codes** | BBT 4106 and BFS 4102                                                                                                                                                                                                                                                                                 |
| **Course Names** | BBT 4206: Business Intelligence I (Week 11-13 of 13) and<br/>BFS 4102: Advanced Business Data Analytics (Week 1-3 of 13)                                                                                                                                                                              |
| **Semester**     | August to November 2026                                                                                                                                                                                                                                                                               |
| **Lecturer**     | Allan Omondi                                                                                                                                                                                                                                                                                          |
| **Contact**      | aomondi@strathmore.edu                                                                                                                                                                                                                                                                                |
| **Note**         | The lecture contains both theory and practice.<br/>This notebook forms part of the practice.<br/>It is intended for educational purposes only.<br/>Recommended citation: [BibTex](https://raw.githubusercontent.com/course-files/RegressionAndClassification/refs/heads/main/RecommendedCitation.bib) |

**Business context**: "Jenga Biashara", is a *fictional* firm focused on financing Kenyan SMEs. It offers loans. Unlike a venture capital investment, which provides capital in exchange for partial ownership of the business, a loan provides money that must be repaid as the principal plus interest. The requests for loans are received from SMEs seeking working capital, equipment financing, expansion funding, or inventory credit. Before a loan officer in Jenga Biashara approves an application, sets an interest rate, or decides whether collateral is required, the underwriting team needs a defensible, data-driven estimate of how risky that applicant is. This risk classification helps the loan officer determine appropriate lending terms, such as the interest rate, monitoring intensity, and, where applicable, collateral requirements. **That makes the model a decision-support tool, rather than an autonomous underwriting system.**

Instead of a single, one-off, accept/reject decision, Jenga Biashara uses a three-tier risk classification: `low`, `medium`, and `high`. This influences how the loan is offered such that:
- the higher the risk classification, the higher the interest rate.
- the higher the risk classification, the greater the level of monitoring required.

The majority of applicants are creditworthy, but a smaller group carries medium risk, and a genuinely risky minority requires the most scrutiny. The *synthetic* dataset provided reflects this common reality (class imbalance) such that roughly 60% are `low` risk, 30% are `medium` risk, and 10% are `high` risk.

The features available to underwriting include hard financial indicators, e.g.
- debt-to-income ratio
- credit score
- missed payment history

They also include soft business signals, e.g.
- online presence
- customer satisfaction
- sector
- county

**Dataset:**
The dataset contains an imperfection:
- **Structural Missingness (as opposed to missing at random):** many working-capital and inventory-financing loans are issued in an unsecured manner, therefore, collateral is not present in the majority of such loan applications


**Learning objectives**:
By the end of this lab, you should be able to:
- detect and quantify class imbalance
- correctly sequence a stratified train/test split relative to preprocessing,
- apply resampling techniques without leaking information into the test set,
- build and compare several classifiers inside a single pipeline,
- read classification-specific diagnostics,
- choose metrics appropriate to an imbalanced multi-class problem,
- tune hyperparameters including class-weight and resampling parameters,
- explain individual predictions, and
- persist a fitted pipeline correctly — including the resampler's train-only behavior.

**Dataset:**

**File**: `sme_credit_risk_kenya.csv` | **Rows**: 2,000 | **Target**: `credit_risk_category` (`Low` / `Medium` / `High`)

| Feature                       | Type                                                            | Description                                                                                                                                                                                                                                                                  |
|-------------------------------|-----------------------------------------------------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `applicant_id`                | Identifier (discrete, ratio scale)                              | Unique reference number for each loan applicant. Not a predictor — excluded from modeling.                                                                                                                                                                                   |
| `sector`                      | Categorical (nominal)                                           | Primary industry of the applicant's business: `Retail`, `Hospitality`, `Agriculture`, `Manufacturing`, `Services`, or `Technology`.                                                                                                                                          |
| `county`                      | Categorical (nominal)                                           | County (or county grouping) where the business operates: `Nairobi`, `Mombasa`, `Kisumu`, `Nakuru`, `Eldoret`, or `Other`.                                                                                                                                                    |
| `quarter`                     | Categorical (nominal)                                           | Calendar quarter in which the loan application was made (`Q1`–`Q4`).                                                                                                                                                                                                         |
| `loan_purpose`                | Categorical (nominal)                                           | Stated reason for the loan: `Working Capital`, `Equipment Purchase`, `Business Expansion`, or `Inventory Financing`.                                                                                                                                                         |
| `owner_education_level`       | Categorical (ordinal)                                           | Highest education level completed by the business owner: `Primary` < `Secondary` < `Certificate` < `Diploma` < `Bachelor's` < `Master's` < `Doctorate`.                                                                                                                      |
| `business_age_years`          | Numeric (continuous, ratio scale)                               | Number of years the applicant's business has been operating. Younger businesses carry structurally higher risk in this dataset, reflecting limited operating history.                                                                                                        |
| `num_employees`               | Numeric (discrete, ratio scale)                                 | Total number of people employed by the business.                                                                                                                                                                                                                             |
| `monthly_revenue_kes`         | Numeric (continuous)                                            | Applicant's reported monthly business revenue, in KES. Right-skewed.                                                                                                                                                                                                         |
| `loan_amount_kes`             | Numeric (continuous, ratio scale)                               | Amount of the loan being applied for, in KES. Right-skewed.                                                                                                                                                                                                                  |
| `loan_term_months`            | Numeric (discrete, ratio scale)                                 | Requested loan repayment term, in months (ranges from 6 to 60).                                                                                                                                                                                                              |
| `interest_rate_pct`           | Numeric (continuous, ratio scale)                               | Interest rate, in percent, quoted for this loan.                                                                                                                                                                                                                             |
| `collateral_value_kes`        | Numeric (continuous, ratio scale)                               | Estimated value of collateral pledged against the loan, in KES. **Missing for a large share of unsecured loans** (`Working Capital` and `Inventory Financing` applications are frequently unsecured) — this is structural missingness, not missing-at-random.                |
| `debt_to_income_ratio`        | Numeric (continuous, bounded)                                   | Ratio of the applicant's existing debt obligations to income. Higher values indicate greater financial strain and are one of the strongest drivers of risk category in this dataset.                                                                                         |
| `credit_score`                | Numeric (continuous, 300–850)                                   | Applicant's credit score, on a standard 300–850 scale. **Missing at random** for roughly 5% of applicants (incomplete credit bureau records).                                                                                                                                |
| `num_previous_loans`          | Numeric (discrete, ratio scale)                                 | Number of loans the applicant has previously taken with this lender or reported elsewhere.                                                                                                                                                                                   |
| `missed_payments_count`       | Numeric (discrete, ratio scale)                                 | Number of missed payments across the applicant's loan history. A direct behavioral risk signal.                                                                                                                                                                              |
| `online_presence_score`       | Numeric (continuous, 0–100)                                     | A composite score reflecting the business's digital visibility (website quality, social media activity, online reviews).                                                                                                                                                     |
| `customer_satisfaction_score` | Numeric (continuous, 1–5)                                       | Average customer satisfaction rating from the business's own customers. **Missing at random** for roughly 4% of applicants.                                                                                                                                                  |
| `competitor_density`          | Numeric (discrete, ratio scale)                                 | Number of directly competing businesses operating in the same immediate area.                                                                                                                                                                                                |
| `credit_risk_category`        | Categorical (ordinal in risk, nominal in encoding) — **Target** | Assigned risk tier: `Low` (~60% of applicants), `Medium` (~30%), `High` (~10%). Determined by an underlying, unobserved risk score combining debt-to-income ratio, missed payments, credit score, loan-to-revenue ratio, collateral coverage, business age, and sector risk. |

**Notes on Synthetic Dataset:**
Two features are synthesized to force specific decisions required to learn specific topics:

- **`collateral_value_kes`** exists for the same reason `avg_daily_foot_traffic` exists in the
  regression dataset: to require you to distinguish *structural* missingness (many
  unsecured loan types genuinely have no collateral to record) from the *random* missingness
  seen in `credit_score` and `customer_satisfaction_score`. Applying one blanket imputation
  rule to all three columns would be a modeling error, not a shortcut.
- **The 60/30/10 class split on `credit_risk_category`** exists to make class imbalance
  unavoidable rather than optional. It determines why the split is stratified,
  why resampling is applied only to the training fold, why cross-validation
  must also be stratified, why precision-recall curves matter more than ROC
  curves for the `High` class, and why accuracy alone is an unreliable metric
  to report to underwriting.

**Remote Environments:**

Do your best to set up your local environment as guided during the lab, however, if you have challenges setting it up, then you can use the following remote environments temporarily for the lab:<br/>

[![Colab](https://img.shields.io/badge/Open-Colab-orange?logo=googlecolab)](
https://colab.research.google.com/github/course-files/RegressionAndClassification/blob/main/2_classification.ipynb) (preferred option)

[![Codespaces](https://img.shields.io/badge/Open-Codespaces-blue?logo=github)](
https://github.com/codespaces/new/course-files/RegressionAndClassification) (alternative)

## Install Dependencies and Import Required Libraries

Install the `ipykernel` package (required for Jupyter notebook) then confirm the following:
1. Where the Python interpreter that will be used to execute code is located
2. The version of the Python interpreter

Then install all the packages into the Jupyter notebook's virtual environment before importing them.

In [ ]:
# Install `ipykernel` for Jupyter notebook
%pip install ipykernel

In [ ]:
# Confirm where the Python interpreter is located
import sys
print(sys.executable)
# print(sys.version)

### Set the Environment Variables

In [ ]:
%pip install python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# This loads .env from the current (or parent) directory if present
# `override=True` ensures that the values set in the .env file take precedence
# over any existing environment variables set in the Operating System
load_dotenv(override=True)

def resolve_environment():
    if os.environ.get("ENVIRONMENT"):
        return os.environ["ENVIRONMENT"].upper()

    # Automatic detection if running in Google Colab
    if "google.colab" in sys.modules:
        return "COLAB"

    # Defaults to a 'DEV' environment if it is not set
    return "DEV"

ENV_SETUP = resolve_environment()

VALID_ENVIRONMENTS = {"PROD", "STAGING", "TESTING", "COLAB", "DEV"}

if ENV_SETUP not in VALID_ENVIRONMENTS:
    print(f"Warning: Invalid ENVIRONMENT '{ENV_SETUP}'. Defaulting to 'DEV'.")
    ENV_SETUP = "DEV"

print(f"Dependencies will be installed for a '{ENV_SETUP}' environment.")

In [ ]:
import sys
import subprocess

def silent_pip_install(args):
    """
    Used to run a command quietly and
    only shows an error if something went wrong.
    :param args: Array of arguments that form the command.
    :return:
    """

    # `sys.executable` is added to ensure you are installing into the virtual environment
    cmd = [sys.executable, "-m", "pip", *args]
    # `subprocess.run` is used to execute an external command
    # `stdout=subprocess.PIPE` captures normal output into result.stdout
    # `stderr=subprocess.PIPE` captures error output into result.stderr
    # `text=True` decodes output as strings (str) instead of bytes, so you can print/read it directly.
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    # `result.returncode` is the process exit status. 0 means success.
    if result.returncode != 0:
        print(result.stderr)

if ENV_SETUP == "PROD":
    silent_pip_install([
        "install", "-r",
        "https://raw.githubusercontent.com/course-files/RegressionAndClassification//refs/heads/main/requirements/prod.txt"
    ])
    print(f"Completed installation of environment dependencies for '{ENV_SETUP}'.")

elif ENV_SETUP in {"STAGING"}:
    silent_pip_install([
        "install", "-r",
        "https://raw.githubusercontent.com/course-files/RegressionAndClassification//refs/heads/main/requirements/dev.txt",
        "-c", "https://raw.githubusercontent.com/course-files/RegressionAndClassification//refs/heads/main/requirements/constraints.txt"
    ])
    print(f"Completed installation of environment dependencies for '{ENV_SETUP}'.")

elif ENV_SETUP in {"TESTING"}:
    silent_pip_install([
        "install", "-r",
        "https://raw.githubusercontent.com/course-files/RegressionAndClassification//refs/heads/main/requirements/dev.txt",
        "-c", "https://raw.githubusercontent.com/course-files/RegressionAndClassification//refs/heads/main/requirements/constraints.txt"
    ])
    print(f"Completed installation of environment dependencies for '{ENV_SETUP}'.")

elif ENV_SETUP in {"COLAB"}:
    silent_pip_install([
        "install", "-r",
        "https://raw.githubusercontent.com/course-files/RegressionAndClassification//refs/heads/main/requirements/colab.txt"
    ])
    print(f"Completed installation of environment dependencies for '{ENV_SETUP}'.")

elif ENV_SETUP in {"DEV"}:
    silent_pip_install([
        "install", "-r",
        "https://raw.githubusercontent.com/course-files/RegressionAndClassification//refs/heads/main/requirements/dev.txt",
        "-c", "https://raw.githubusercontent.com/course-files/RegressionAndClassification//refs/heads/main/requirements/constraints.txt"
    ])
    print(f"Completed installation of environment dependencies for '{ENV_SETUP}'.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# For suppressing warnings displayed in the notebook
import warnings
warnings.filterwarnings('ignore')

print("The environment ready.")

In [ ]:
# Uncomment the line below if any of these packages are not yet installed
# %pip install pandas numpy scikit-learn matplotlib seaborn imbalanced-learn shap lime joblib scipy statsmodels -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# For suppressing warnings displayed in the notebook
import warnings
warnings.filterwarnings('ignore')

print("The environment is ready.")

## Load the Data

We load the synthetic SME dataset and confirm its shape (number of
rows/observations and number of columns/features), data types, and target
variable before proceeding further. This enables us to understand the structure
of the dataset and identify any potential issues that may need to be addressed
before training the model.

In [ ]:
# For file and system operations
import urllib.request
import os

dataset_path = './data/sme_credit_risk_kenya.csv'
url = 'https://raw.githubusercontent.com/course-files/RegressionAndClassification/refs/heads/main/data/sme_credit_risk_kenya.csv'

if not os.path.exists(dataset_path):
    print("Downloading dataset...")
    if not os.path.exists('./data'):
        os.makedirs('./data')
    urllib.request.urlretrieve(url, dataset_path)
    print("✅ Dataset downloaded")
else:
    print("✅ Dataset already exists locally")

use_cols = ['applicant_id','sector','county','quarter','loan_purpose','owner_education_level','business_age_years','num_employees','monthly_revenue_kes','loan_amount_kes','loan_term_months','interest_rate_pct','collateral_value_kes','debt_to_income_ratio','credit_score','num_previous_loans','missed_payments_count','online_presence_score','customer_satisfaction_score','competitor_density','credit_risk_category']
sme_data = pd.read_csv(dataset_path, usecols=use_cols, encoding='utf-8', nrows=200000)

In [ ]:
print(f"Shape: {sme_data.shape[0]} rows, {sme_data.shape[1]} columns")
sme_data.head()

In [ ]:
sme_data.info()

### Identify the Target Variable

In [ ]:
TARGET = "credit_risk_category"

# `applicant_id` is an identifier, not a predictive feature. We will drop it
# but keep it in the raw (orig
print(f"Target variable: {TARGET}")
print(f"Classes: {sorted(sme_data[TARGET].unique())}")
print(f"Number of predictor candidates (excluding id and target): {sme_data.shape[1] - 2}")

## Initial Exploratory Data Analysis

The initial Exploratory Data Analysis (EDA) is composed of:

### Measures of Frequency
  - Count (number of observations [rows], number of features [columns])
  - Percent (for categorical features)

### Measures of Central Tendency
  - Mean
  - Median
  - Mode

### Measures of Relationship
  - Correlation (both Spearman and Pearson)

### Measures of Distribution
  - Minimum value
  - Lower quartile (`Q1`)
  - Middle quartile (`Q2`) = Median
  - Upper quartile (`Q3`)
  - Maximum value
  - Variance
  - Standard deviation
  - Skewness
  - Kurtosis
  - Interquartile range (IQR) = `Q3 - Q1`
  - Lower fence
  - Upper fence

Reference image:
![BoxCox](https://raw.githubusercontent.com/course-files/RegressionAndClassification/refs/heads/main/assets/images/BoxPlot.jpg)

#### Interpretation of Kurtosis (Fisher kurtosis)

Kurtosis describes "tail heaviness" relative to a normal distribution.

Interpretation of kurtosis (Fisher kurtosis):
1. Kurtosis < 0: platykurtic (lighter tails than normal)
2. Kurtosis near 0: mesokurtic (similar tail weight to normal)
3. Kurtosis > 0: leptokurtic (heavier tails than normal)

Higher kurtosis (>0) indicates more extreme values may occur, which can affect
methods sensitive to outliers. Lower kurtosis (<0) indicates lighter tails and
fewer extreme values than a normal distribution.

Common remedies when heavy tails are problematic include robust methods, transformation
(e.g., log1p, Box-Cox, Yeo-Johnson), winsorization, or careful outlier handling.

#### Interpretation of Skewness (Fisher-Pearson sample skewness)

Skewness measures the asymmetry of a distribution around its mean.

Interpretation of skewness (Fisher-Pearson sample skewness):
1. Skewness < 0 indicates a negative (left) skew.
2. Skewness near 0 indicates approximate symmetry.
3. Skewness > 0 indicates a positive (right) skew.

Practical rule-of-thumb bands (context-dependent):
- Between -0.5 and 0.5: approximately symmetric
- 0.5 to 1.0 (or -0.5 to -1.0): moderate skew
- Above 1.0 (or below -1.0): strong skew

**Note:** symmetry does not by itself imply a normal (Gaussian) distribution.

High skew can make mean-based summaries and some model assumptions less
reliable.

Common remedies include transformations (log1p, square-root, Box-Cox,
Yeo-Johnson) or robust methods/algorithms.

### 3.1 Measures of Distribution

We check class balance first, before anything else. This single check determines which
metrics will be trustworthy later and whether resampling is required.

In [ ]:
class_counts = sme_data[TARGET].value_counts()
# `normalize=True` returns proportions (fractions/percent share) instead of raw counts.
class_proportions = sme_data[TARGET].value_counts(normalize=True).round(3)

class_summary = pd.DataFrame({"count": class_counts, "proportion": class_proportions})
class_summary

**Notes on Modelling Decisions:**

The `High` risk class makes up roughly 10% of applicants. **This is not a data
quality issue. It reflects real credit portfolios, where high-risk defaults are the
minority outcome.** Any model that predicts `Low` for every applicant would already
achieve close to 60% accuracy while being useless for its actual purpose (identifying
risk). This is a key factor that will be considered in subsequent steps.

In [ ]:
numeric_cols = sme_data.select_dtypes(include=[np.number]).columns.drop("applicant_id")
categorical_cols = sme_data.select_dtypes(exclude=[np.number, 'datetime64[ns]']).columns

print("\nThe identified numeric columns (measures) are:")
print(numeric_cols.tolist())

print("\nThe identified categorical columns (dimensions) are:")
print(categorical_cols.tolist())

q1 = sme_data[numeric_cols].quantile(0.25)
q2 = sme_data[numeric_cols].quantile(0.50)
q3 = sme_data[numeric_cols].quantile(0.75)

iqr = q3 - q1

distribution_summary = pd.DataFrame({
    "min": sme_data[numeric_cols].min(),
    "q1_25%": q1,
    "q2_50%": q2,
    "q3_75%": q3,
    "max": sme_data[numeric_cols].max(),
    "mean": sme_data[numeric_cols].mean(),
    "mode": sme_data[numeric_cols].mode().iloc[0],
    "variance": sme_data[numeric_cols].var(),
    "std_dev": sme_data[numeric_cols].std(),
    "skewness": sme_data[numeric_cols].skew(),
    "kurtosis": sme_data[numeric_cols].kurt(),
    "IQR": iqr,
    "lower_fence": q1 - 1.5 * iqr,
    "upper_fence": q3 + 1.5 * iqr,
})

distribution_summary.round(2).style.format("{:,.2f}")

### 3.2 Measures of Relationship

#### The Analysis of Variance (ANOVA) F-statistic

The Analysis of Variance (ANOVA) F-statistic tests whether the mean of a
numeric predictor differs significantly across several target classes.
It works by comparing the variance between class groups to the variance
within each class group.

It requires data that does not contain missing values; therefore, we use a
simple median data imputation.

**Interpreting the F-statistic and p-value:**

**F-statistic:** Measures how much a feature's mean differs across the target
classes relative to variation within each class. A higher value indicates
stronger separation between classes; a value near zero indicates little
separation. We prioritize features with a high ANOVA F-statistic as strong
considerations for the model and give it closer attention during feature
selection.

**p-value:** The probability of observing the F-statistic (or a larger one)
if the feature had no real relationship with the target. A value **below 0.05**
is generally taken as evidence of a genuine association; a higher value
suggests the difference could be due to chance.

This test detects only linear separation between group means and does not
account for interactions between features. It is used as an initial screen,
not a final decision.

In [ ]:
from sklearn.feature_selection import f_classif

numeric_predictors = sme_data[numeric_cols].copy()
# Temporary median fill purely for this diagnostic (not the pipeline imputation)
numeric_predictors_filled = numeric_predictors.fillna(numeric_predictors.median())

f_stats, p_values = f_classif(numeric_predictors_filled, sme_data[TARGET])
anova_summary = pd.DataFrame({
    "feature": numeric_cols,
    "F_statistic": f_stats,
    "p_value": p_values
}).sort_values("F_statistic", ascending=False)
anova_summary.round(4)

#### Cramer's V

**What it measures:** The strength of association between a categorical
predictor and the categorical target, based on the chi-square statistic.

**Range:** From 0 to 1, regardless of the number of categories in
either variable, which makes it comparable across different categorical
features.

**Interpretation of magnitude:**
- **0.00 – 0.10:** Negligible association
- **0.10 – 0.20:** Weak association
- **0.20 – 0.40:** Moderate association
- **0.40 and above:** Strong association

**Important caveats:**
- Cramer's V indicates association strength only; it does not indicate the
direction of the relationship, because categorical variables have no inherent
order.
- A high value with a category that has very few observations should be
treated with caution, because small cell counts can produce an unstable
chi-square statistic.

Cramer's V, like the F-statistic, evaluates each categorical feature
against the target in isolation. Two features, such as "sector" and
"loan_purpose," may overlap substantially in the information they carry, and a
high Cramer's V for both does not mean both deserve a place in the final model
together.

We use this alongside the ANOVA results to build intuition before formal
feature selection.

In [ ]:
# Cramer's V: association strength between categorical predictors and the target
def cramers_v(confusion_matrix):
    # A contingency table is a grid that displays the count of observations
    # for every combination of two categorical variables. Each row represents
    # a category of one variable, each column represents a category of the
    # other, and each cell holds the number of observations that fall into
    # that particular row-column combination.
    chi2 = stats.chi2_contingency(confusion_matrix)[0]   # chi-square statistic for the contingency table
    n = confusion_matrix.sum().sum()                     # total number of observations
    phi2 = chi2 / n                                       # normalize chi-square by sample size
    r, k = confusion_matrix.shape                         # r = number of rows (categories in predictor), k = number of columns (categories in target)

    # Bias correction (Bergsma's correction), since phi2, r, and k tend to overestimate association in small samples
    phi2_corrected = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    r_corrected = r - ((r - 1) ** 2) / (n - 1)
    k_corrected = k - ((k - 1) ** 2) / (n - 1)

    # Final Cramer's V, bounded between 0 and 1
    return np.sqrt(phi2_corrected / min(k_corrected - 1, r_corrected - 1))

categorical_cols = [col for col in categorical_cols if col != TARGET]
cramers_v_results = {}

for col in categorical_cols:
    # Build a contingency table: counts of each category vs. each target class
    contingency = pd.crosstab(sme_data[col], sme_data[TARGET])
    cramers_v_results[col] = cramers_v(contingency.values)

# Collect results into a single table, sorted so the strongest associations appear first
cramers_v_df = pd.DataFrame.from_dict(cramers_v_results, orient="index", columns=["Cramers_V"])
cramers_v_df.sort_values("Cramers_V", ascending=False).round(3)

In [ ]:
plt.figure(figsize=(12, 9))
corr_matrix = sme_data[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix Among Numeric Predictors")
plt.tight_layout()
plt.show()

### 3.3 Basic Visualization

In [ ]:
plt.figure(figsize=(7, 4.5))
sns.countplot(data=sme_data, x=TARGET, order=["Low", "Medium", "High"], color="#4F6EA4")
plt.title("Class Balance: Credit Risk Category")
plt.ylabel("Number of Applicants")
plt.tight_layout()
plt.show()

In [ ]:
key_predictors = ["debt_to_income_ratio", "missed_payments_count", "credit_score",
                   "business_age_years", "loan_amount_kes", "num_previous_loans"]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, col in zip(axes.flatten(), key_predictors):
    sns.boxplot(data=sme_data, x=TARGET, y=col, order=["Low", "Medium", "High"], ax=ax, color="#4F6EA4")
    ax.set_title(f"{col} by Risk Category")
plt.tight_layout()
plt.show()

In [ ]:
pairplot_cols = ["debt_to_income_ratio", "missed_payments_count", "credit_score", TARGET]
sns.pairplot(sme_data[pairplot_cols].sample(500, random_state=RANDOM_STATE), hue=TARGET,
             hue_order=["Low", "Medium", "High"], diag_kind="kde", plot_kws={"alpha": 0.4, "s": 15})
plt.suptitle("Pairplot Colored by Risk Category (sample of 500 rows)", y=1.02)
plt.show()

## Train/Test Split

We stratify on `credit_risk_category` so that the rare `High` risk class (~10%) is
represented proportionally in both the training and test sets. Without stratification, a
random split could easily leave the test set with very few `High` risk examples, making
the performance metrics unstable.

**This step occurs before feature selection and data preprocessing or data
transformation stages.** Fitting selectors, encoders, scalers, or resamplers
on the full dataset before splitting leaks information into the test set.

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [c for c in sme_data.columns if c not in ["applicant_id", TARGET]]

X = sme_data[feature_cols].copy()
y = sme_data[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")
print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTest class proportions:")
print(y_test.value_counts(normalize=True).round(3))

## 5. Feature Selection *(performed on training data only)*

All feature selection below uses `X_train` / `y_train` exclusively.

### 5.1 Chi-square test (categorical predictors)

The chi-square test requires non-negative integer-encoded categories and is sensitive to
low expected cell counts, so we apply it to label-encoded categorical columns.

- **Chi2 statistic:** Measures the strength of association between a feature
and the target. Higher values indicate a stronger relationship, meaning the
feature's distribution differs more between classes than would be expected by
chance.
- **p-value:** The probability of observing this association (or a stronger
one) if the feature were actually independent of the target. Low p-values
(typically below 0.05) suggest the association is unlikely to be due to random
noise, so the feature is probably informative.

**For feature selection:** Rank by chi2_statistic, retain features with high
statistics and low p-values, and discard features with high p-values (weak or
no real association).

In [ ]:
print(f"The starting features are {len(X_train.columns)}: {list(X_train.columns)}")

In [ ]:
from sklearn.feature_selection import chi2, SelectKBest, mutual_info_classif
from sklearn.preprocessing import LabelEncoder

categorical_train = X_train[categorical_cols].copy()
categorical_train_encoded = categorical_train.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)))

chi2_stats, chi2_p_values = chi2(categorical_train_encoded, y_train)
chi2_summary = pd.DataFrame({
    "feature": categorical_cols,
    "chi2_statistic": chi2_stats,
    "p_value": chi2_p_values
}).sort_values("chi2_statistic", ascending=False)
chi2_summary.round(4)

### 5.2 ANOVA F-test and Mutual Information (numeric predictors)

**Key limitation of ANOVA F-test (`f_classif`):** It only detects linear,
mean-based differences. A feature could have identical means across classes but
a completely different spread or shape per class, and ANOVA would miss it
entirely.

**Mutual Information (MI) (`mutual_info_classif`)**

Measures how much knowing the feature's value reduces your uncertainty about
the target, **regardless of the shape of that relationship**. It captures
linear relationships, non-linear relationships, and even non-monotonic patterns.

A relationship is **non-monotonic** if it changes direction at some point:
rising then falling until the end, or falling then rising until the end. Example:
`age` and `reaction time` in athletic performance often rises through childhood
and adolescence, peaks in early adulthood, then declines from age 24 onwards.

**High MI score:** Feature and target are strongly dependent, in any pattern.
**MI score near zero:** Feature carries little to no information about the target.

**Key advantage over ANOVA:** It will catch relationships ANOVA cannot, but,
at the cost of being noisier and less stable on small datasets, because it is
estimated rather than calculated with precision.

How to use ANOVA and Mutual Information together for feature selection

| Situation                    | 	Interpretation                                                                        |
|------------------------------|---------------------------------------------------------------------------------------|
| High F-statistic AND high MI | Strong, likely linear signal; safe to retain                                          |
| Low F-statistic but high MI  | Non-linear relationship ANOVA missed; worth retaining, consider non-linear algorithms |
| High F-statistic but low MI  | Rare; check for a data leakage or coding issue                                        |
| Low on both                  | Weak candidate. Consider removing it.                                                 |

Mutual information captures non-linear dependence that the ANOVA F-test (a linear,
mean-difference test) can miss. Where the two disagree substantially on ranking, prefer
mutual information as a sanity check, but do not discard a feature on the strength of
mutual information alone if it has a clear domain rationale.

In [ ]:
numeric_train = X_train[numeric_cols].copy()
numeric_train_filled = numeric_train.fillna(numeric_train.median())

f_stats_train, p_values_train = f_classif(numeric_train_filled, y_train)
mi_scores = mutual_info_classif(numeric_train_filled, y_train, random_state=RANDOM_STATE)

numeric_selection_summary = pd.DataFrame({
    "feature": numeric_cols,
    "ANOVA_F": f_stats_train,
    "p_value": p_values_train,
    "mutual_info": mi_scores
}).sort_values("mutual_info", ascending=False)
numeric_selection_summary.round(4)

### Recursive Feature Elimination (RFE)

RFE fits a model repeatedly, removing the weakest predictor(s) at each
iteration. We use a simple linear model as the estimator and rank all numeric
features (categorical features are handled separately via the encoding pipeline).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFECV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

# Standardization is used to
scaler_for_rfe = StandardScaler()
numeric_train_scaled = scaler_for_rfe.fit_transform(numeric_train_filled)

rfe_estimator = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)

# `StratifiedKFold` is used rather than plain `KFold` because your target is
# categorical. Stratification ensures each fold preserves the original class
# proportions, which matters for classification, particularly if any class is
# imbalanced.

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rfecv_selector = RFECV(
    estimator=rfe_estimator,
    step=1,
    cv=cv_strategy,
    scoring="accuracy",
    min_features_to_select=1
)
rfecv_selector.fit(numeric_train_scaled, y_train)

rfecv_ranking = pd.DataFrame({
    "feature": numeric_cols,
    "selected": rfecv_selector.support_,
    "rank": rfecv_selector.ranking_
}).sort_values("rank")

print(f"Optimal number of features: {rfecv_selector.n_features_}")
rfecv_ranking

### Tree-Based Feature Importance as a Filter

**What It Is**

Tree-based models, e.g., Random Forest, Gradient Boosting, and so on, build
their predictions by repeatedly splitting the data on feature values. Features
that create the most useful splits which separate the classes are used more often and earlier in the trees. This usage pattern is converted into an **importance score** for each feature.

**How the Score is Calculated**

Two common methods exist:

1. **Mean Decrease in Impurity (MDI)** — the default in scikit-learn (`feature_importances_`). Measures how much each feature reduces impurity (Gini or entropy) across all the splits where it is used, averaged across all trees.
2. **Permutation Importance** — measures the drop in model performance when a single feature's values are randomly shuffled. If shuffling a feature barely affects model performance, then it imples that that feature was not being relied upon.

**Key distinction:** MDI is fast but calculated on the training data, so it can be biased toward high-cardinality features (those with many unique values). Permutation importance is slower but more reliable, and it can be calculated on a held-out test set.

**How to Interpret the Scores**

- **High importance**: The feature was frequently useful for splitting the data into purer class groups; likely a strong predictor.
- **Low importance**: The feature was rarely selected for splitting; likely weak or redundant.
- Scores across all features typically sum to 1 (a proportion of total importance), so they are best read as *relative* rankings, not as importance in any absolute sense.

**Why Use It as a "Filter"**

Rather than trusting one tree's importance ranking directly for final feature selection, it is used as a **quick, cheap screening step**:

1. Train a tree-based model (often Random Forest, for stability) on all candidate features.
2. Rank features by importance.
3. Discard the bottom tier of consistently low-importance features.
4. Carry the remaining shortlist forward into more rigorous selection methods, or into the final model itself.

**Key Strengths**

- Captures **non-linear relationships** and **interactions between features** automatically, unlike ANOVA or correlation.
- Requires little preprocessing; no need to scale features first.
- Works natively on a mix of numeric and categorical (encoded) data.

**Key Limitations, Tell it Like it is**

- **Biased toward high-cardinality features**: A feature with many unique values (an ID column, for example) can appear artificially important simply because it offers more possible split points, not because it is genuinely predictive.
- **Unstable with correlated features**: If two features are highly correlated, the model may arbitrarily favour one and assign the other low importance, even though both carry similar information. Removing the "unimportant" one based on a single run can be a mistake.
- **Importance depends on the specific model run**: Results can shift somewhat with different random seeds. Averaging importance across multiple runs, or multiple trees within a Random Forest, produces a more stable ranking.

**Quick Comparison to Earlier Methods**

| Method                | Detects Non-Linearity? | Detects Interactions? | Speed            |
|-----------------------|------------------------|-----------------------|------------------|
| Chi2 / ANOVA          | No                     | No                    | Fast             |
| Mutual Information    | Yes                    | No                    | Moderate         |
| Tree-Based Importance | Yes                    | Yes                   | Moderate to Slow |

---

Tree-based importance answers "how useful was this feature to *this particular model*," not "how useful is this feature in general." A feature deemed unimportant by a Random Forest might become highly important in a linear model, or vice versa. Treating tree-based importance as an absolute, model-agnostic truth about a feature's real-world value, rather than as one model's opinion, is a common and consequential error. It is a good filter for early triage; it is not the final word on a feature's worth.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_filter = RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=300, max_depth=8)
rf_filter.fit(numeric_train_filled, y_train)

tree_importance = pd.DataFrame({
    "feature": numeric_cols,
    "importance": rf_filter.feature_importances_
}).sort_values("importance", ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=tree_importance, y="feature", x="importance", color="#4F6EA4")
plt.title("Tree-Based Feature Importance (Random Forest filter)")
plt.tight_layout()
plt.show()

**Notes on Modelling Decisions**: all four selection methods agree that `debt_to_income_ratio`,
`missed_payments_count`, `loan_amount_kes`, `monthly_revenue_kes`, and `credit_score` are
strong predictors, consistent with domain expectation (these directly encode repayment
capacity and history). No columns are hard-dropped at this stage; the full feature set
proceeds to the next stage because tree-based models can tolerate weaker predictors,
and the selection evidence above is retained for the Explainability discussion in later stages.
`applicant_id` was already excluded as a non-predictive identifier.

## Data Preprocessing and Data Transformation *(performed on training data only)*

**Data Transformation Techniques:**

Data transforms can improve the accuracy of your final model when applied
before the modelling stage. It is standard practice to apply multiple
transforms. Data transforms can be grouped into the following 3 categories:

- Basic data transforms:
  - **Scaling:** Divides each value by the standard deviation
  - **Centering:** Subtracts the mean from each value
  - **Standardization:** Ensures that each numeric attribute has a
     mean value of 0 and a standard deviation of 1. This is done
     by combining the scale data transform and the centre data
     transform: subtract the mean and divide by the standard deviation.
  - **Normalization:** Ensures the numerical data are between [0, 1]
     (inclusive).

- Power data transforms:
  - **Box-Cox:** reduces the skewness by shifting the distribution of
  an attribute and making the attribute have a more
  Gaussian-like distribution.
  - **Yeo-Johnson:** like Box-Cox, Yeo-Johnson reduces the skewness
  by shifting the distribution of an attribute and making the
  attribute have a more Gaussian-like distribution.
  The difference is that Yeo-Johnson can handle zero and
  negative values.

- Logarithmic data transforms:
  - **log1p: logarithm of (1 + x):** Applies a logarithmic transformation that
  helps reduce right-skewness and compress large values for non-negative data.
  It can also handle zero values, making it useful for variables such as revenue or counts.

**Note:** All data transformations are fit on `X_train` / `y_train` only and
applied to `X_test` via `.transform()`. They are never refit on the test data
to avoid leakage.

**Notes on Modelling Decisions:**

### Encoding

- One-hot encoding for nominal categoricals (`sector`, `county`, `quarter`, `loan_purpose`);
- Ordinal encoding for `owner_education_level` (natural order: `Primary` < `Secondary` < `Certificate` < `Diploma` < `Bachelor's` < `Master's` < `Doctorate`).

### Handling Missing Data

- `collateral_value_kes`: missing for a large share of unsecured loans (structural, not
  random). We apply **median imputation** plus a **missingness indicator**, so that the model can use
  **"no collateral on file"** as informative.
- `credit_score`, `customer_satisfaction_score`: missing at random. We also use median imputation here.

### Class Imbalance

We compare **SMOTE**, **ADASYN**, **random undersampling**, and **class-weighting** as four
distinct strategies.
All resampling techniques are applied **inside the pipeline, fit on the training
fold only** (never fit on the test set, and never allowed to touch test data even during
cross-validation) using `imbalanced-learn`'s pipeline, which is leakage-safe by
construction: `Pipeline.fit()` resamples, but `Pipeline.predict()` does not.

### Scaling / Centering / Standardization / Normalization

`StandardScaler` for models sensitive to feature magnitude (Logistic Regression, KNN, SVM);
included in the shared pipeline for all models for comparability.

### Box-Cox / Yeo-Johnson

`monthly_revenue_kes` and `loan_amount_kes` are right-skewed and strictly positive. This makes them good Box-Cox
candidates. `debt_to_income_ratio` is bounded and does not need this transform.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, PowerTransformer
from sklearn.impute import SimpleImputer

nominal_cols = ["sector", "county", "quarter", "loan_purpose"]
ordinal_col = ["owner_education_level"]
# OrdinalEncoder expects a list of category lists: one category list for each ordinal column.
# Reminder:
#   list_example = [1, 2, 3, "four"]
#   array_example = np.array([1, 2, 3, 4])
#   dictionary_example = {"key1": "value1", "key2": "value2"}
education_order = [["Primary", "Secondary", "Certificate", "Diploma", "Bachelors", "Masters", "Doctorate"]]

collateral_col = ["collateral_value_kes"]
other_numeric_cols = [c for c in numeric_cols if c not in collateral_col]

collateral_pipeline = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

numeric_pipeline = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

nominal_pipeline = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

ordinal_pipeline = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=education_order)),
])

preprocessor = ColumnTransformer(transformers=[
    ("collateral", collateral_pipeline, collateral_col),
    ("numeric", numeric_pipeline, other_numeric_cols),
    ("nominal", nominal_pipeline, nominal_cols),
    ("ordinal", ordinal_pipeline, ordinal_col),
])

print("Preprocessing pipeline constructed.")
print(f"Collateral (structural missingness + indicator): {collateral_col}")
print(f"Other numeric columns ({len(other_numeric_cols)}): {other_numeric_cols}")
print(f"Nominal columns (one-hot): {nominal_cols}")
print(f"Ordinal column: {ordinal_col}")

In [ ]:
# Box-Cox / Yeo-Johnson demonstration on skewed predictors
boxcox_demo = X_train[["monthly_revenue_kes"]].copy()
boxcox_transformer = PowerTransformer(method="box-cox", standardize=False)
boxcox_result = boxcox_transformer.fit_transform(boxcox_demo)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.histplot(boxcox_demo["monthly_revenue_kes"], kde=True, ax=axes[0], color="#4F6EA4")
axes[0].set_title("monthly_revenue_kes: Before Box-Cox")

sns.histplot(boxcox_result.flatten(), kde=True, ax=axes[1], color="#4F6EA4")
axes[1].set_title("monthly_revenue_kes: After Box-Cox")

plt.tight_layout()
plt.show()

print(f"Skewness before: {boxcox_demo['monthly_revenue_kes'].skew():.2f}")
print(f"Skewness after Box-Cox: {pd.Series(boxcox_result.flatten()).skew():.2f}")

### Resampling Techniques

Resampling techniques are designed to address the problem of imbalanced classes
in the target variable, where one class significantly outnumbers another. This
is common in fraud detection, loan default, and rare disease diagnosis type of
datasets. If it is not corrected, most models will learn to favor the majority
class simply because doing so minimizes average error, often at the cost of
correctly identifying the minority class that actually matters.

Resampling techniques include:

#### Random Undersampling:
Randomly removes examples from the majority class
until the class distribution is more balanced. Simple and fast, but it discards
data, which can remove useful information and increase the risk of underfitting,
particularly if the majority class was not heavily oversized to begin with.

#### SMOTE (Synthetic Minority Oversampling Technique):
Generates new, synthetic examples of the minority class rather than simply duplicating
existing ones. It does this by selecting a minority-class point, finding its
nearest minority-class neighbors, and creating new points along the line between
them. This avoids the overfitting risk of naive duplication but can create
unrealistic examples if the minority class is sparse or noisy. This is because
it interpolates blindly between points without regard to whether that space
makes real-world sense.

#### ADASYN (Adaptive Synthetic Sampling): A refinement of SMOTE.
Rather than generating synthetic examples uniformly across the minority class, it generates
more synthetic examples in regions where the minority class is hardest to
learn, specifically, near the boundary with the majority class. This focuses
effort where the model needs it most, but for the same reason, it can be more
sensitive to noisy or mislabeled boundary points than standard SMOTE.

#### Class-Weighting: Rather than altering the dataset itself, this adjusts
the model's loss function to penalize misclassification of the minority class
more heavily than misclassification of the majority class. It is typically
applied with a single parameter (`class_weight="balanced"` in scikit-learn) and
requires no synthetic data generation or removal of real data. This makes it
generally safer and more efficient than resampling, though it depends on the
chosen model supporting a weighted loss function, and it does not address the
minority class's information sparsity the way synthetic oversampling attempts
to.

**Note:** All resampling is fit and applied on `X_train` / `y_train` only.
`X_test` is left in its original, untouched distribution, since the test set
must reflect real-world, class proportions to give an honest measure of
performance.

**Resampling the test set, or fitting a resampler before the train/test split,
is a common and serious source of data leakage.**

In [ ]:
# Comparing resampling strategies -- fit on training fold only, for illustration here
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler

# Preprocess a copy of the training data so that we can visualize class balance
# after resampling
X_train_transformed_preview = preprocessor.fit_transform(X_train, y_train)

resamplers = {
    "Original (no resampling)": None,
    "SMOTE": SMOTE(random_state=RANDOM_STATE),
    "ADASYN": ADASYN(random_state=RANDOM_STATE),
    "Random Undersampling": RandomUnderSampler(random_state=RANDOM_STATE),
}

resample_comparison = {}
for name, resampler in resamplers.items():
    if resampler is None:
        resample_comparison[name] = y_train.value_counts()
    else:
        _, y_resampled = resampler.fit_resample(X_train_transformed_preview, y_train)
        resample_comparison[name] = pd.Series(y_resampled).value_counts()

resample_comparison_df = pd.DataFrame(resample_comparison).reindex(["Low", "Medium", "High"])
resample_comparison_df

Note that `preprocessor.fit_transform(X_train, y_train)` above is used only to
preview resampling behavior. **Inside the actual modeling pipeline in the next
stage, preprocessing and resampling are refit within each cross-validation
fold**, not fit once on the whole training set, so that no fold's validation
portion influences the resampler fit for that fold.

## Machine Learning Pipeline and Cross-Validation

We use `imblearn.pipeline.Pipeline` (**NOT** `sklearn.pipeline.Pipeline`) because it supports
resampling steps that are correctly skipped at prediction time. Each model is compared
using **stratified** k-fold cross-validation, which preserves class proportions in every
fold. Stratified k-fold cross-validation is necessary given the 60/30/10 class imbalance.

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=15),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=6),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=300, max_depth=8),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE, n_estimators=200, max_depth=3),
    "Support Vector Machine": SVC(probability=True, random_state=RANDOM_STATE),
    "Naive Bayes": GaussianNB(),
}

stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, model in models.items():
    pipeline = ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", model),
    ])
    scores = cross_val_score(pipeline, X_train, y_train, cv=stratified_kfold,
                              scoring="f1_macro", n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:25s} F1-macro: {scores.mean():.4f}  (+/- {scores.std():.4f})")

In [ ]:
cv_results_df = pd.DataFrame(cv_results)
plt.figure(figsize=(11, 5))
sns.boxplot(data=cv_results_df, color="#4F6EA4")
plt.ylabel("Stratified Cross-Validated F1-macro")
plt.title("Stratified 5-Fold Cross-Validation: F1-macro by Model (with SMOTE)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Why we use F1-macro for model selection here**: F1-macro averages the F1-score across all
three classes with equal weight, so the model cannot achieve a high score merely by
performing well on the majority `Low` class while ignoring `High`. This is discussed further
alongside weighted averages in a later stage.

## Diagnostic Exploratory Data Analysis (Model Diagnostics) *(classification-specific)*

The diagnostics below use the Random Forest and Logistic Regression pipelines, fit once on the
full training set (with SMOTE applied only to the training fold, as configured in the
pipeline itself).

In [ ]:
class_labels = ["Low", "Medium", "High"]

rf_pipeline = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=300, max_depth=8)),
])
rf_pipeline.fit(X_train, y_train)

logreg_pipeline = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
logreg_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)
y_pred_logreg = logreg_pipeline.predict(X_test)

### Confusion Matrix

**What It Is**

A confusion matrix is a table that compares a model's predicted classes against
the actual, true classes. It takes the following form for a binary classification problem:

|                     | Predicted Negative  | Predicted Positive  |
|---------------------|---------------------|---------------------|
| **Actual Negative** | True Negative (TN)  | False Positive (FP) |
| **Actual Positive** | False Negative (FN) | True Positive (TP)  |

**The Four Core Terms**

- **True Positive (TP):** Model predicted positive, and it was actually positive. A correct catch.
- **True Negative (TN):** Model predicted negative, and it was actually negative. A correct pass.
- **False Positive (FP):** Model predicted positive, but it was actually negative. A false alarm, also called a **Type I error**.
- **False Negative (FN):** Model predicted negative, but it was actually positive. A missed case, also called a **Type II error**.

**Metrics Derived From a Confusion Matrix**

- **Accuracy** = `(TP + TN) / Total`: The overall proportion that is correct. This can be misleading on imbalanced data, since predicting the majority class every time can still score high.
- **Precision** = TP / (TP + FP) — Of everything predicted positive, what proportion was actually positive. High precision means few false alarms.
- **Recall (Sensitivity)** = TP / (TP + FN) — Of everything actually positive, what proportion was correctly caught. High recall means few missed cases.
- **F1 Score** = harmonic mean of precision and recall — A single balance point between the two, useful when both false positives and false negatives carry a real cost.
- **Specificity** = TN / (TN + FP) — Of everything actually negative, what proportion was correctly identified as negative.

**How to Read a Confusion Matrix**

1. Look at the diagonal (TP and TN) first; this is what the model predicted correctly.
2. Look at the off-diagonal (FP and FN) next; this is where the model is failing, and the two failure types are not interchangeable.
3. Ask yourself which error type is more costly for the specific problem you are addressing. In loan default prediction, a false negative, that is, missing an actual default, is typically more expensive than a false positive, that is, flagging a safe borrower as risky. In spam detection, the reverse is often true; a false positive, that is, sending a genuine email to spam, is usually worse than missing one spam message.
4. Choose the metric to optimize for based on that cost asymmetry, rather than defaulting to accuracy out of habit.

**Summary Reference Table**

| Metric    | Formula                               | Answers                                       |
|-----------|---------------------------------------|-----------------------------------------------|
| Accuracy  | (TP+TN)/Total                         | How often is the model right, overall?        |
| Precision | TP/(TP+FP)                            | When it predicts positive, can it be trusted? |
| Recall    | TP/(TP+FN)                            | Of all real positives, how many were found?   |
| F1        | Harmonic mean of Precision and Recall | Balance of the precision and recall           |


**Confusion Matrix for a Multi-Class Classification Problem**

Rather than a 2x2 table, the matrix expands to an N x N grid, where N is the number of classes. Each row represents an actual class, and each column represents a predicted class. The matrix therefore has one row and one column per class, not just "positive" and "negative."

|          | 	Pred A  | 	Pred B  | 	Pred C  |
|----------|---------|---------|---------|
| Actual A | 	Correct	 | Error   | Error   |
| Actual B	 | Error   | Correct	 | Error   |
| Actual C	 | Error   | Error   | Correct |

**How to Read a Multi-Class Confusion Matrix**

The diagonal still matters the most.

The diagonal cells (top-left to bottom-right) represent correct predictions for each class, exactly as a 2x2 matrix. Everything off the diagonal is a misclassification, but now there is no single "false positive" or "false negative" cell; instead, there is a full breakdown of exactly which class is being **confused** with which other class. This is the main advantage of the multi-class confusion matrix: it shows where the model is failing, not merely that it is failing.

**How to Read It**

1. Check the diagonal first, as with binary classification, to see overall correctness per class.
2. Scan each row for the largest off-diagonal value; this identifies which specific class is most often confused with which other specific class.
3. Look for asymmetry in the confusion. A model might confuse Class A with Class B frequently, while rarely confusing Class B with Class A; this pattern matters more than a single averaged score can reveal.
4. Choose macro average when all classes must be taken seriously regardless of frequency (a rare disease category, for example, cannot be allowed to be poor simply because it is small), and weighted average when overall dataset-wide performance is the genuine goal.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (name, y_pred) in zip(axes, [("Random Forest", y_pred_rf), ("Logistic Regression", y_pred_logreg)]):
    cm = confusion_matrix(y_test, y_pred, labels=class_labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix: {name}")
plt.tight_layout()
plt.show()

### Receiver Operating Characteristic (ROC) Curve

**What It Is**

The ROC (Receiver Operating Characteristic) curve plots a model's performance across every possible classification threshold, not just the default 0.5. It shows the trade-off between correctly catching positives and incorrectly flagging negatives as the threshold shifts.

**The Two Axes**
- **Y-axis:** True Positive Rate (TPR), also called Recall or Sensitivity: TP / (TP + FN). Of all actual positives, what proportion was caught?
- **X-axis:** False Positive Rate (FPR): FP / (FP + TN). Of all actual negatives, what proportion was wrongly flagged as positive?

As the threshold is lowered, more cases get classified as positive, so both TPR and FPR rise together. The curve traces this trade-off across the full range of thresholds, from 0 to 1.

**How to Read the Shape**

- Curve hugging the top-left corner: Strong model. High TPR achieved while FPR stays low.
- Diagonal line (from bottom-left to top-right): No better than random guessing.
- Curve below the diagonal: Worse than random; a serious problem, often indicating inverted labels somewhere in the pipeline.

**AUC (Area Under the Curve)**

A single summary number, from 0 to 1, representing the entire curve:

- AUC = 1.0: Perfect separation between classes.
- AUC = 0.5: No better than random guessing.
- AUC < 0.5: Worse than random guessing.
Interpreted as: the probability that the model ranks a randomly chosen actual positive higher than a randomly chosen actual negative.

**Practical Use**

Use ROC/AUC to compare models' overall ranking ability, independent of any single chosen threshold.
Use the curve itself to pick a specific threshold that balances TPR and FPR according to the actual cost of each error type in the given problem.

**Note:**  ROC-AUC can look excellent even on badly imbalanced data, because FPR is calculated against the large majority class in the denominator, which stays easy to keep low almost by default. **On a highly imbalanced problem, the Precision-Recall curve is typically better** at measuring performance than ROC-AUC, since it evaluates performance directly against the minority class, which is usually the class that matters most.

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_test_binarized = label_binarize(y_test, classes=class_labels)
y_pred_logreg = logreg_pipeline.predict_proba(X_test)

fig, ax = plt.subplots(figsize=(7, 6))
for i, cls in enumerate(class_labels):
    fpr, tpr, _ = roc_curve(y_test_binarized[:, i], y_pred_logreg[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{cls} (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set_xlabel("False Positive Rate (FPR)")
ax.set_ylabel("True Positive Rate (TPR)")
ax.set_title("ROC Curve (One-vs-Rest): Logistic Regression")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_test_binarized = label_binarize(y_test, classes=class_labels)
y_proba_rf = rf_pipeline.predict_proba(X_test)

fig, ax = plt.subplots(figsize=(7, 6))
for i, cls in enumerate(class_labels):
    fpr, tpr, _ = roc_curve(y_test_binarized[:, i], y_proba_rf[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{cls} (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set_xlabel("False Positive Rate (FPR)")
ax.set_ylabel("True Positive Rate (TPR)")
ax.set_title("ROC Curve (One-vs-Rest): Random Forest")
ax.legend()
plt.tight_layout()
plt.show()

### Precision-Recall Curve (PR-AUC)

**What It Is**

Plots a model's performance across **every possible classification threshold**, similar to the ROC curve, but using precision and recall instead of TPR and FPR. It focuses entirely on the positive class, making it more informative than ROC on imbalanced data.

**The Two Axes**

- **Y-axis — Precision:** TP / (TP + FP). Of everything predicted positive, what proportion was actually positive?
- **X-axis — Recall:** TP / (TP + FN). Of everything actually positive, what proportion was correctly caught?

As the threshold is lowered, recall rises (more positives get caught), but precision typically falls (more false alarms creep in). The curve traces this trade-off directly.

**How to Read the Shape**

- **Curve hugging the top-right corner:** Strong model. High precision maintained even at high recall.
- **Curve dropping sharply as recall increases:** The model can only maintain precision at low recall; it struggles to catch more positives without a heavy cost in false alarms.
- **Baseline (a flat horizontal line):** Equal to the proportion of positives in the dataset. This is the performance of a model that predicts positive at random; a genuinely useful model's curve should sit clearly above this line, not the diagonal used in ROC.

**AUC-PR (Area Under the Precision-Recall Curve)**

A single summary number representing the whole curve; higher is better, with the meaningful floor being the baseline rate of positives, not 0.5 as with ROC-AUC.

**Practical Use**

- Preferred over ROC/AUC when the positive class is rare, since it does not get inflated by a large, easy-to-classify negative class.
- Use the curve to pick a threshold that balances the specific cost of a false positive against the specific cost of a false negative for the problem at hand.

**Comparison to ROC**

|                              | ROC Curve                | Precision-Recall Curve    |
|------------------------------|--------------------------|---------------------------|
| Axes                         | TPR vs FPR               | Precision vs Recall       |
| Sensitive to class imbalance | No (can be misleading)   | Yes, by design            |
| Baseline                     | Diagonal (0.5)           | Positive class proportion |
| Best used when               | Classes roughly balanced | Positive class is rare    |

**Note:** because the precision-recall baseline changes with the actual proportion of positives in the dataset, AUC-PR values are not directly comparable across two different datasets or even two different train/test splits with different class balances, unlike ROC-AUC's fixed 0.5 baseline. A precision-recall curve should always be read alongside its own baseline, and comparisons of AUC-PR across differently balanced samples should be treated with real caution, not taken at face value.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(7, 6))
for i, cls in enumerate(class_labels):
    precision, recall, _ = precision_recall_curve(y_test_binarized[:, i], y_pred_logreg[:, i])
    ap_score = average_precision_score(y_test_binarized[:, i], y_pred_logreg[:, i])
    ax.plot(recall, precision, label=f"{cls} (AP = {ap_score:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve (One-vs-Rest): Logistic Regression")
ax.legend()
plt.tight_layout()
plt.show()

print("Note the 'High' risk class curve: this is the rare, business-critical class, and its")
print("PR-AUC is the more honest indicator of usefulness than its ROC-AUC would suggest.")

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(7, 6))
for i, cls in enumerate(class_labels):
    precision, recall, _ = precision_recall_curve(y_test_binarized[:, i], y_proba_rf[:, i])
    ap_score = average_precision_score(y_test_binarized[:, i], y_proba_rf[:, i])
    ax.plot(recall, precision, label=f"{cls} (AP = {ap_score:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve (One-vs-Rest): Random Forest")
ax.legend()
plt.tight_layout()
plt.show()

print("Note the 'High' risk class curve: this is the rare, business-critical class, and its")
print("PR-AUC is the more honest indicator of usefulness than its ROC-AUC would suggest.")

### Calibration Curve (Reliability Diagram)

**What It Is**

A calibration curve, also called a reliability diagram, checks whether a model's predicted probabilities actually match real-world likelihoods. A model can have excellent accuracy or AUC while still producing probability estimates that are systematically wrong, and this is precisely what a calibration curve is designed to reveal.

**How It Is Built**

1. Predictions are sorted and grouped into bins based on predicted probability, for example, all predictions between 0.7 and 0.8.
2. For each bin, the actual observed fraction of positives is calculated.
3. Predicted probability (x-axis) is plotted against observed fraction of positives (y-axis) for each bin.

**How to Read It**

- **Perfectly calibrated model:** Points fall along the diagonal line (y = x). A predicted probability of 0.7 means that, among all cases given that prediction, roughly 70 percent were actually positive.
- **Curve above the diagonal:** Model is **under-confident**. Actual positive rate is higher than predicted, meaning the model is too cautious.
- **Curve below the diagonal:** Model is **over-confident**. Actual positive rate is lower than predicted, meaning the model overstates its certainty.
- **S-shaped curve:** A common pattern, particularly with models like SVMs or tree ensembles, where predictions near 0 and 1 are pushed too close to the extremes, while mid-range predictions are compressed toward 0.5.

**Why It Matters**

Accuracy, precision, recall, and AUC all evaluate whether the *ranking* or *classification* of predictions is correct; none of them check whether the actual probability values are trustworthy. Calibration matters specifically whenever a raw probability, not just a class label, will be used downstream, for example, in risk scoring, expected loss calculations, or any decision-making process where "how confident" matters as much as "which class."

**Fixing Poor Calibration**

- **Platt Scaling:** Fits a logistic regression on top of the model's raw scores to rescale them; works well with small datasets and models with a sigmoid-like miscalibration (such as SVMs).
- **Isotonic Regression:** A more flexible, non-parametric rescaling method; works well with larger datasets, but can overfit on small ones.

**Note:** the resampling techniques covered earlier, SMOTE, ADASYN, and undersampling in particular, actively distort calibration, since they change the class balance the model is trained on relative to the real-world balance it will face. A model trained on artificially balanced data will typically look badly over-confident on the minority class once it is applied to real, imbalanced data, even if its raw classification performance improved. Calibration should always be checked after any resampling step, not assumed to be unaffected by it.

In [ ]:
# Calibration curve -- checks whether predicted probabilities are trustworthy
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 6))
for i, cls in enumerate(class_labels):
    prob_true, prob_pred = calibration_curve(y_test_binarized[:, i], y_pred_logreg[:, i], n_bins=8)
    ax.plot(prob_pred, prob_true, marker="o", label=cls)
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfectly calibrated")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration Curve (One-vs-Rest): Logistic Regression")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Calibration curve -- checks whether predicted probabilities are trustworthy
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 6))
for i, cls in enumerate(class_labels):
    prob_true, prob_pred = calibration_curve(y_test_binarized[:, i], y_proba_rf[:, i], n_bins=8)
    ax.plot(prob_pred, prob_true, marker="o", label=cls)
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfectly calibrated")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration Curve (One-vs-Rest): Random Forest")
ax.legend()
plt.tight_layout()
plt.show()

| Term             | Meaning                                                                                                                                                    |
|------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Precision**    | Out of all items the model said are class X, how many are actually class X?                                                                                |
| **Recall**       | Out of all actual items in class X, how many did the model correctly find?                                                                                 |
| **F1-score**     | A balance between precision and recall, such that a higher value means better balance.                                                                     |
| **Support**      | The number of actual items in that class.                                                                                                                  |
| **Macro avg**    | The average of precision, recall, and F1-score across all classes, treating each class equally regardless of size.                                         |
| **Weighted avg** | The average of precision, recall, and F1-score, weighted by each class's support, so classes with more samples have greater influence on the final figure. |

If a rare minority class is the entire point of the model, e.g., fraud, loan default, rare disease, then **macro average** is the honest figure to report, since weighted average will happily mask poor minority-class performance behind strong majority-class numbers.
Presenting the weighted average alone on an imbalanced problem is a common way to make a genuinely weak model look presentable.

In [ ]:
# Class-wise error breakdown
from sklearn.metrics import classification_report
import numpy as np
from sklearn.metrics import classification_report

y_pred_labels = logreg_pipeline.predict(X_test)

print("Logistic Regression classification report:\n")

print(classification_report(y_test, y_pred_labels, zero_division=0))

In [ ]:
# Class-wise error breakdown
from sklearn.metrics import classification_report

print("Random Forest classification report:\n")
print(classification_report(y_test, y_pred_rf, labels=class_labels))

## Evaluating the Performance of Several Models

All models are fit on the training set (with SMOTE applied to the training fold via the
pipeline) and evaluated once on the untouched test set.

**Accuracy alone is misleading here**: a model predicting `Low` for every applicant would
score close to 60% accuracy while providing zero value for identifying risk. We, therefore,
report macro and weighted precision/recall/F1 alongside accuracy, plus ROC-AUC, PR-AUC, and
log loss.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, log_loss)

evaluation_results = []

for name, model in models.items():
    pipeline = ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", model),
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)

    y_test_bin = label_binarize(y_test, classes=class_labels)

    evaluation_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Precision (weighted)": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall (macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall (weighted)": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1 (macro)": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "F1 (weighted)": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "ROC-AUC (ovr, macro)": roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr"),
        "PR-AUC (macro)": np.mean([average_precision_score(y_test_bin[:, i], y_proba[:, i])
                                     for i in range(len(class_labels))]),
        "Log Loss": log_loss(y_test, y_proba, labels=class_labels),
    })

evaluation_df = pd.DataFrame(evaluation_results).sort_values("F1 (macro)", ascending=False)
evaluation_df.round(4)

In [ ]:
best_model_name = evaluation_df.iloc[0]["Model"]
print(f"Best model on held-out test set (by F1-macro): {best_model_name}")
print("\nCompare 'F1 (macro)' against 'Accuracy' row by row: models with high accuracy but")
print("noticeably lower F1-macro are likely performing well on 'Low' while struggling on")
print("'High' -- exactly the failure mode accuracy alone would hide.")

## Hyperparameter Tuning

We tune the best-performing candidate from the previous stage, searching jointly over model
hyperparameters, the resampling strategy, and `class_weight`, using stratified
cross-validation.

In [ ]:
from sklearn.model_selection import GridSearchCV

tuning_pipeline = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("resampler", SMOTE(random_state=RANDOM_STATE)),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE)),
])

param_grid = {
    "resampler": [SMOTE(random_state=RANDOM_STATE), ADASYN(random_state=RANDOM_STATE),
                  RandomUnderSampler(random_state=RANDOM_STATE)],
    "model__n_estimators": [200, 300],
    "model__max_depth": [6, 8, 12],
    "model__class_weight": [None, "balanced"],
}

grid_search = GridSearchCV(
    tuning_pipeline, param_grid, cv=stratified_kfold,
    scoring="f1_macro", n_jobs=-1, verbose=0
)
grid_search.fit(X_train, y_train)

print(f"Best tuned model: {grid_search.best_estimator_.named_steps['model']}")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV F1-macro: {grid_search.best_score_:.4f}")

In [ ]:
best_tuned_model = grid_search.best_estimator_
y_pred_tuned = best_tuned_model.predict(X_test)

tuned_f1_macro = f1_score(y_test, y_pred_tuned, average="macro")
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)

print(f"Tuned model -- Test Accuracy: {tuned_accuracy:.4f} | Test F1-macro: {tuned_f1_macro:.4f}")
print("\nWe then compare this against the untuned Random Forest to assess")
print("whether tuning (including the resampling-strategy search) improved generalization.")

## Explainability

### Logistic Regression coefficients as odds ratios

In [ ]:
# Fit a binary framing for clean odds-ratio interpretation: High risk vs. not High risk
y_train_binary = (y_train == "High").astype(int)
y_test_binary = (y_test == "High").astype(int)

logreg_binary_pipeline = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
logreg_binary_pipeline.fit(X_train, y_train_binary)

feature_names_out = preprocessor.get_feature_names_out()
log_odds = logreg_binary_pipeline.named_steps["model"].coef_[0]
odds_ratios = np.exp(log_odds)

odds_ratio_df = pd.DataFrame({
    "feature": feature_names_out,
    "log_odds_coefficient": log_odds,
    "odds_ratio": odds_ratios
}).sort_values("odds_ratio", ascending=False)

print("Interpretation: an odds ratio > 1 means the feature increases the odds of 'High' risk;")
print("an odds ratio < 1 means it decreases the odds, holding other features constant.")
odds_ratio_df.head(10)

### 11.2 Tree-based feature importance (Random Forest, all three classes)

In [ ]:
final_feature_names = preprocessor.get_feature_names_out()
rf_importance = pd.DataFrame({
    "feature": final_feature_names,
    "importance": rf_pipeline.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False)

plt.figure(figsize=(9, 7))
sns.barplot(data=rf_importance.head(15), y="feature", x="importance", color="#4F6EA4")
plt.title("Random Forest Feature Importance (Top 15)")
plt.tight_layout()
plt.show()

### SHapley Additive exPlanations (SHAP) Values

SHAP is a method for explaining individual predictions of a machine learning
model. It answers the question: "For this specific prediction, how much did
each feature contribute, and in which direction?"

**Origin:** It is built on Shapley values, a concept from cooperative game
theory originally designed to fairly distribute a payout among players based
on their individual contribution to a team effort **(Shapley, 1952)**. SHAP
repurposes this idea: features are treated as "players," and the model's
prediction (relative to a baseline/average prediction) is the "payout" to be
distributed.

**Core idea:** For a given prediction, SHAP computes each feature's contribution
by considering all possible orderings in which features could be "added" to
the model and averaging the marginal effect of that feature across all these
orderings. This ensures the contribution attributed to each feature is fair and
consistent, instead of being dependent on an arbitrary order.

**Key properties:**

- *Additive:* If you add up the SHAP values of every feature for a given
observation, the total equals exactly the difference between that
observation's prediction and the model's baseline (average) prediction.
No portion of the prediction is left unexplained.

- *Local:* SHAP produces a separate explanation for each individual observation,
rather than one explanation for the model as a whole. As a result, the same
feature can contribute differently to different observations, even though the
underlying model does not change.

- *Model-agnostic in principle:* SHAP can be applied to any model (linear
regression, random forests, gradient boosting, neural networks), although
model-specific implementations (e.g., TreeExplainer for tree-based models)
are faster than the generic version of SHAP.

**Expected output:**

- *Force plot / waterfall plot*: Shows one prediction. Start at the baseline
(average prediction), then see how each feature pushes it up or down, bar by
bar, until you reach the final predicted value.

- *Summary plot (beeswarm)*: Shows the whole dataset. Features are ranked top
to bottom by importance. Each dot is one observation. Its left/right position
shows whether that feature pushed the prediction down or up, and its color (red
= high value, blue = low value) shows whether high or low values of that feature
drive predictions up or down.

**Caveat:** **SHAP values explain what the model learned, not necessarily true
causal relationships in the real world**. This means that a feature with a
large SHAP value is influential to the model's output, not automatically the
"true cause" of the outcome.

Shapley, L. S. (1952). *A Value for N-Person Games.* RAND Corporation.
[https://doi.org/10.7249/P0295](https://doi.org/10.7249/P0295)

In [ ]:
import shap

X_test_transformed = rf_pipeline.named_steps["preprocessor"].transform(X_test)
explainer = shap.TreeExplainer(rf_pipeline.named_steps["model"])
shap_values = explainer.shap_values(X_test_transformed[:300])

# For a multi-class model, SHAP returns either a list of per-class arrays (older versions)
# or a single 3D array shaped (n_samples, n_features, n_classes) (newer versions).
# The code below handles both, so it does not break as the library version changes.
print(f"Model classes in order: {rf_pipeline.named_steps['model'].classes_}")
high_class_index = list(rf_pipeline.named_steps["model"].classes_).index("High")

if isinstance(shap_values, list):
    shap_values_high_class = shap_values[high_class_index]
else:
    shap_values_high_class = shap_values[:, :, high_class_index]

shap.summary_plot(shap_values_high_class, X_test_transformed[:300],
                   feature_names=final_feature_names, show=False)
plt.title("SHAP Summary: Drivers of 'High' Risk Classification")
plt.tight_layout()
plt.show()

### Local Interpretable Model-agnostic Explanations (LIME) for individual predictions

While SHAP explains feature contributions across the whole model, LIME explains a single
prediction by fitting a local, interpretable surrogate model around that one instance. This
is useful for explaining an individual loan decision to a non-technical stakeholder.

LIME is a method for explaining individual predictions of a machine learning model. It answers the question: "For this specific prediction, what simple, local relationship between features and outcome would approximate the model's behaviour here?"

**Origin:** Introduced as a general-purpose technique for explaining "black box" model predictions in an interpretable and faithful manner **(Ribeiro, Singh, & Guestrin, 2016)**. The core insight is that a complex model may be impossible to understand globally, but its behaviour in the small neighbourhood around one specific data point can often be approximated well by a much simpler model.

**Core idea:** For a given prediction, LIME generates a new dataset of perturbed samples by slightly altering the feature values of the observation being explained. It then obtains the complex model's predictions for each of these perturbed samples and fits a simple, interpretable model, typically a weighted linear regression, on this new local dataset, weighting samples closer to the original observation more heavily. The simple model's coefficients become the explanation, since a linear model is easy to read directly.

**Key properties:**

- *Local:* LIME explains one prediction at a time, and makes no claim about the model's behaviour anywhere outside the immediate neighbourhood of that specific observation. The same feature can carry a different weight in the explanation for a different observation, even under the same underlying model.

- *Model-agnostic:* LIME treats the underlying model purely as a black box, querying it for predictions on perturbed samples without needing access to its internal structure. This makes it usable on genuinely any model, including ones with no available gradient or tree structure.

- *Approximate, not exact:* Unlike SHAP's additive property, LIME's local linear model is a genuine approximation of the complex model's behaviour, not a mathematically exact decomposition of the prediction. Two separate runs of LIME on the same prediction can produce somewhat different explanations, since the perturbation and sampling process involves randomness.

**Expected output:**

- *Feature weight list / bar chart*: Shows one prediction. Lists the features that most influenced that specific prediction, alongside a weight indicating both the direction (positive or negative) and the relative strength of each feature's local contribution.

- *Highlighted input (text/image tasks)*: For text or image models, LIME commonly highlights the specific words or image regions that contributed most strongly to the prediction, since perturbation naturally maps to removing or masking words and superpixels.

**Caveat:** **LIME's explanation is only as trustworthy as the local linear approximation it relies upon.** If the model's true decision boundary is highly non-linear even within the small local neighbourhood being sampled, a linear approximation can be a poor fit, and the explanation produced can be unstable or misleading despite looking clean and confident.

Ribeiro, M. T., Singh, S., and Guestrin, C. (2016). *"Why Should I Trust You?": Explaining the Predictions of Any Classifier.* Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining, 1135–1144. [https://doi.org/10.1145/2939672.2939778](https://doi.org/10.1145/2939672.2939778)

**Note:** SHAP and LIME are frequently presented as interchangeable tools that answer the same question, and they are not. SHAP's additive property gives a mathematically consistent, globally comparable attribution; LIME's local surrogate model gives a fast, intuitive, but genuinely approximate explanation that can shift between runs on the identical prediction. When the two tools are applied to the same observation and disagree meaningfully on which feature mattered most, the disagreement itself is a valuable signal, often indicating that the model's true local behaviour is complex enough that neither explanation should be accepted uncritically on its own.

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    training_data=rf_pipeline.named_steps["preprocessor"].transform(X_train),
    feature_names=list(final_feature_names),
    class_names=list(rf_pipeline.named_steps["model"].classes_),
    mode="classification",
    random_state=RANDOM_STATE,
)

sample_instance = X_test_transformed[0]
lime_explanation = lime_explainer.explain_instance(
    sample_instance,
    rf_pipeline.named_steps["model"].predict_proba,
    num_features=8,
)

print(f"Explaining prediction for test applicant index {X_test.index[0]}")
print(f"Actual risk category: {y_test.iloc[0]}")
for feature, weight in lime_explanation.as_list():
    print(f"  {feature:65s} {weight:+.4f}")

## Model Persistence

We persist the entire fitted `imblearn` pipeline (preprocessing, resampler, and model) as a single object.

**Important distinction**: the resampler (SMOTE, ADASYN, or undersampling) is a
*train-only* step. `imblearn.pipeline.Pipeline` enforces this automatically: calling
`.fit()` on the pipeline resamples the training data before fitting the model, but calling
`.predict()` does **not** invoke the resampler at all. New, unseen applicants are never
synthetically resampled.

This is the exact behavior we want (resampling exists to
rebalance the training distribution the model learns from, not to alter real applicants at
inference time).

In [ ]:
import joblib
import os
from datetime import datetime

# Automatically picks the best tuned pipeline from GridSearchCV
final_pipeline = grid_search.best_estimator_

# (Optional) refit on full training data to be explicit; often already fit by GridSearchCV refit=True
final_pipeline.fit(X_train, y_train)

# Ensure the model folder exists
model_dir = './model'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

# Build filename with model name and timestamp
model_name = final_pipeline.named_steps["model"].__class__.__name__
# This timestamp pattern is useful beyond this one script.
# It prevents accidentally overwriting a previous run's saved model.
# It also helps you to keep track of model versions.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_filename = f"{timestamp}_{model_name}_classifier_for_sme_credit_risk.joblib".lower()
model_path = os.path.normpath(os.path.join(model_dir, model_filename))

joblib.dump(final_pipeline, model_path)
print("Selected model:", model_name)
print(f"✅Best pipeline (pre-processing steps + transformation steps + model) saved to: {model_path}")

In [ ]:
# Demonstrate that the resampler does not run at inference time
loaded_pipeline = joblib.load("./model/randomforestclassifier_classifier_for_sme_credit_risk.joblib")

new_applicant = pd.DataFrame([{
    "sector": "Agriculture",
    "county": "Nakuru",
    "quarter": "Q2",
    "loan_purpose": "Working Capital",
    "owner_education_level": "Secondary",
    "business_age_years": 1.5,
    "num_employees": 3,
    "monthly_revenue_kes": 180000,
    "loan_amount_kes": 450000,
    "loan_term_months": 24,
    "interest_rate_pct": 18.5,
    "collateral_value_kes": np.nan,  # unsecured working capital loan
    "debt_to_income_ratio": 0.72,
    "credit_score": 560,
    "num_previous_loans": 3,
    "missed_payments_count": 2,
    "online_presence_score": 20.0,
    "customer_satisfaction_score": 3.1,
    "competitor_density": 9,
}])

predicted_class = loaded_pipeline.predict(new_applicant)[0]
predicted_proba = loaded_pipeline.predict_proba(new_applicant)[0]

print(f"Predicted risk category: {predicted_class}")
print("\nClass probabilities:")
for cls, proba in zip(loaded_pipeline.named_steps["model"].classes_, predicted_proba):
    print(f"  {cls}: {proba:.3f}")

print(f"\nSingle applicant predicted -- output row count: 1 (confirms the resampler,")
print("which only ever increases or decreases row counts during training, did not")
print("fire during inference: exactly one input row produced exactly one prediction).")

In [ ]:
# Second test case: a lower-risk applicant profile, to contrast with the higher-risk
# unsecured working-capital applicant already demonstrated in the notebook.
loaded_pipeline = joblib.load("./model/randomforestclassifier_classifier_for_sme_credit_risk.joblib")

new_applicant_2 = pd.DataFrame([{
    "sector": "Manufacturing",
    "county": "Nairobi",
    "quarter": "Q4",
    "loan_purpose": "Equipment Purchase",
    "owner_education_level": "Masters",
    "business_age_years": 9.0,
    "num_employees": 22,
    "monthly_revenue_kes": 1250000,
    "loan_amount_kes": 600000,
    "loan_term_months": 36,
    "interest_rate_pct": 13.0,
    "collateral_value_kes": 950000,  # equipment purchase loans are typically secured
    "debt_to_income_ratio": 0.21,
    "credit_score": 740,
    "num_previous_loans": 2,
    "missed_payments_count": 0,
    "online_presence_score": 65.0,
    "customer_satisfaction_score": 4.4,
    "competitor_density": 4,
}])

predicted_class_2 = loaded_pipeline.predict(new_applicant_2)[0]
predicted_proba_2 = loaded_pipeline.predict_proba(new_applicant_2)[0]

print(f"Predicted risk category: {predicted_class_2}")
print("\nClass probabilities:")
for cls, proba in zip(loaded_pipeline.named_steps["model"].classes_, predicted_proba_2):
    print(f"  {cls}: {proba:.3f}")

print(f"\nSingle applicant predicted -- output row count: 1 (confirms the resampler did")
print("not fire during inference, as in the first example above).")

print("\nComparison with the first applicant:")
print("  Applicant 1 (Agriculture, unsecured working capital, DTI 0.72, credit score 560,")
print("  2 missed payments) -- profile consistent with elevated risk.")
print("  Applicant 2 (Manufacturing, secured equipment loan, DTI 0.21, credit score 740,")
print("  0 missed payments) -- profile consistent with lower risk.")
print("  If the model's predicted classes do not reflect this ordering, then that is worth")
print("  investigating before trusting the model on real applications.")

## Summary

This notebook walked through a complete classification workflow on an imbalanced,
three-class credit risk problem: exploratory analysis with explicit class-balance checking,
a stratified leakage-safe split, feature selection performed exclusively on training data
using four complementary methods, a preprocessing pipeline handling encoding, structured and
random missingness, scaling, and skew correction, resampling strategy comparison, stratified
cross-validated comparison of seven classifiers, classification-specific diagnostics
(confusion matrix, ROC, precision-recall, calibration), imbalance-aware evaluation metrics,
joint hyperparameter and resampling-strategy tuning, four explainability techniques, and
persistence of the complete pipeline with an explicit demonstration that the resampler does
not act at inference time.

**A note on the core idea of this lab**: every design decision after the Initial EDA traces back
to the same fact: the target is imbalanced. This shaped the split (stratified), the
preprocessing (resampling), the cross-validation (stratified folds), the diagnostics
(precision-recall over ROC alone), and the evaluation metric of choice (F1-macro over raw
accuracy). **Recognizing this thread is more valuable than memorizing any single technique in
isolation.**